# Session 4 exercise, a one-sample audit test

> **A client's ledger reports an average daily transaction value of 520.**
> **You sample 40 actual trading days. Is 520 supported by the evidence, or overstated?**

This is a one-sample test: one group, compared against a single claimed value. You will state the hypotheses, run the test, read the p-value against alpha, and confirm the verdict with a confidence interval. Work through it, then push Claude on the reflection at the end.

## Step 0, the sample

Load the 40 sampled days. These are the actual recorded daily averages the audit pulled.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(21)
sample = np.round(rng.normal(472, 78, 40), 2)   # the 40 sampled days (EUR)
claimed = 520.0                                 # the client's reported average
alpha = 0.05

print(f'n           = {sample.size}')
print(f'sample mean = {sample.mean():,.2f}')
print(f'sample SD   = {sample.std(ddof=1):,.2f}')
print(f'claimed     = {claimed:,.2f}')

plt.figure(figsize=(8, 3.2))
plt.hist(sample, bins=12, color='#0D73B8', alpha=0.8)
plt.axvline(claimed, color='#C0392B', lw=2, ls='--', label=f'claimed {claimed:,.0f}')
plt.axvline(sample.mean(), color='#3E8E5A', lw=2, label=f'sample mean {sample.mean():,.0f}')
plt.xlabel('daily transaction value (EUR)'); plt.ylabel('days'); plt.legend()
plt.tight_layout(); plt.show()

## Step 1, state the hypotheses

Before touching the data, write them down. The null is the status quo claim with an equality; the alternative is what you would need evidence to conclude.

- **H0:** the true average daily transaction value equals 520.  `mu = 520`
- **Ha:** it differs from 520.  `mu =/= 520`  (two-sided)

We will reject H0 only if the sample would be surprising when 520 is true.

## Step 2, run the one-sample t-test

The population SD is unknown, so this is a t-test. `scipy.stats.ttest_1samp` does it in one line and returns the test statistic and the two-sided p-value.

In [ ]:
t_stat, p_value = stats.ttest_1samp(sample, popmean=claimed)
print(f't statistic = {t_stat:.3f}')
print(f'p-value     = {p_value:.4f}')

## Step 3, the verdict

Compare the p-value with alpha. If `p < alpha`, the data is too surprising for the claim to stand, so reject H0.

In [ ]:
if p_value < alpha:
    print(f'p = {p_value:.4f} < {alpha}: reject H0.')
    direction = 'below' if sample.mean() < claimed else 'above'
    print(f'The sample mean ({sample.mean():,.2f}) sits {direction} the claimed 520.')
    print('The reported average is not supported by the sample: flag it for review.')
else:
    print(f'p = {p_value:.4f} >= {alpha}: fail to reject H0.')
    print('No evidence the true average differs from the claimed 520.')

## Step 4, confirm with a confidence interval

A 95% confidence interval should tell the same story: the claimed value falls outside the interval exactly when the two-sided test rejects it at the same level.

In [ ]:
mean, sd, n = sample.mean(), sample.std(ddof=1), sample.size
se = sd / np.sqrt(n)
tstar = stats.t.ppf(0.975, df=n - 1)
lo, hi = mean - tstar * se, mean + tstar * se
inside = lo <= claimed <= hi
print(f'95% CI for the true average: {lo:,.2f} to {hi:,.2f}')
print(f'claimed 520 is {"inside" if inside else "OUTSIDE"} the interval')
print('The interval and the test agree.')

## Stretch and your call

1. **Halve the margin.** The margin of error is `t* x SD / sqrt(n)`. Roughly how large a sample would you need to halve it, holding the SD fixed? Compute it.
2. **One side or two?** The audit cares mainly whether the client *overstated* the average. Would a one-sided test change the p-value, and would it change the verdict here?
3. **Significant, but how much?** The gap here is about 48 per day. Is that material? How does the sample size change whether a small gap turns out significant?
4. **Type I vs Type II.** If you flag this client and the 520 was actually fine, which error did you make, and what did it cost? Which error would worry an audit partner more?